In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch as t
from transformer_lens import HookedTransformer
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
from dadapy.data import Data
from collections import defaultdict


import transformer_lens.utils as utils
import einops

from joblib import Parallel, delayed
import pandas as pd

import plot_utils

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
# Saves computation time, since we don't need it for the contents of this notebook
t.set_grad_enabled(False)


In [3]:
# Load GPT-2 Small
model = HookedTransformer.from_pretrained("gpt2-small")

# Load prompt from Pile-10K (Prompt 3218)
pile_dataset = load_dataset("NeelNanda/pile-10k")

Loaded pretrained model gpt2-small into HookedTransformer


In [4]:
filtered_indices = np.load('filtered_indices.npy')


In [5]:
filtered_indices

array([   0,   19,   22, ..., 9989, 9991, 9998], dtype=int64)

In [6]:
pile_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'meta'],
        num_rows: 10000
    })
})

In [7]:
filtered_dataset = pile_dataset['train'][filtered_indices]

In [8]:
setname_to_indexlist = defaultdict(list)

In [9]:
for i, set_name in enumerate(filtered_dataset['meta']):
    setname_to_indexlist[set_name['pile_set_name']].append(i)

In [10]:
arxiv_indices, wiki_indices, math_indices = setname_to_indexlist['ArXiv'], setname_to_indexlist['Wikipedia (en)'], setname_to_indexlist['DM Mathematics']

In [11]:
arxiv_prompts, wiki_prompts, math_prompts = [filtered_dataset['text'][idx] for idx in arxiv_indices], [filtered_dataset['text'][idx] for idx in wiki_indices], [filtered_dataset['text'][idx] for idx in math_indices]


In [12]:
def prompts_to_tokens(prompts):
    tokens = model.to_tokens(prompts, prepend_bos=True)
    tokens = tokens[..., :512]
    return tokens

In [13]:
arxiv_tokens, wiki_tokens, math_tokens = prompts_to_tokens(arxiv_prompts), prompts_to_tokens(wiki_prompts), prompts_to_tokens(math_prompts)

In [14]:
# Function to compute intrinsic dimension (ID) with dadapy
def compute_ids(full_reps):
    ids = []
    for rep in full_reps:
        _data = Data(coordinates=rep.cpu().detach().numpy(), maxk=100)
        ids.append(_data.return_id_scaling_gride(range_max=64))
    return np.array(ids)

In [15]:
def zero_attn_out_hook(attn_out, hook):
    # print(attn_out.shape)
    return t.zeros_like(attn_out)

zero_attn_hooks = [
            # Changed hook point to blocks.{layer}.attn.hook_result
            (utils.get_act_name("attn_out", layer), zero_attn_out_hook)
            for layer in range(model.cfg.n_layers)
        ]

In [16]:
    idim_accumulated = []
    idim_accumulated_free = []
    
    
    for idx in range(len(arxiv_tokens[:N])):
        ids = compute_ids(accumulated_residual[:, idx])
        ids_free = compute_ids(accumulated_residual_free[:, idx])
        idim_accumulated.append(ids[:, 0, 1])
        idim_accumulated_free.append(ids_free[:, 0, 1])
    # print(accumulated_residual.shape)
    return labels, idim_accumulated, idim_accumulated_free

NameError: name 'N' is not defined

In [33]:
logits, cache = model.run_with_cache(arxiv_tokens[0])
accumulated_residual, labels = cache.decompose_resid(layer = -1,\
                                            return_labels = True)

In [51]:
labels

['embed',
 'pos_embed',
 '0_attn_out',
 '0_mlp_out',
 '1_attn_out',
 '1_mlp_out',
 '2_attn_out',
 '2_mlp_out',
 '3_attn_out',
 '3_mlp_out',
 '4_attn_out',
 '4_mlp_out',
 '5_attn_out',
 '5_mlp_out',
 '6_attn_out',
 '6_mlp_out',
 '7_attn_out',
 '7_mlp_out',
 '8_attn_out',
 '8_mlp_out',
 '9_attn_out',
 '9_mlp_out',
 '10_attn_out',
 '10_mlp_out',
 '11_attn_out',
 '11_mlp_out']

In [17]:
def tokens_to_idim_prompt_and_free_decomposed_attn_heads(tokens, N):
    per_head_residuals = []
    per_layer_residuals = [] 
    logit_list = []
    
    dataloader = t.utils.data.DataLoader(tokens[:N], batch_size=1, shuffle=False)
    for batch in dataloader:      
        logits, cache = model.run_with_cache(batch)
        logit_list.append(logits)
        accumulated_residual, labels = cache.accumulated_resid(layer = -1, incl_mid=True,\
                                                    return_labels = True)
        per_layer_residual, labels = cache.decompose_resid(
            layer=-1, return_labels=True
        )
        per_layer_residuals.append(per_layer_residual)
        with model.hooks(fwd_hooks=zero_attn_hooks):
            logits_free, cache_free = model.run_with_cache(batch)
            accumulated_residual_free, _ = cache_free.accumulated_resid(layer = -1, incl_mid=True,\
                                                    return_labels = True)
            
        per_head_residual, labels = cache.stack_head_results(
            layer=-1, return_labels=True, incl_remainder=True
        )
        per_head_residuals.append(per_head_residual)
        
        
    logit_cat = t.cat(logit_list, dim=0)    
    loss = model.loss_fn(logit_cat, arxiv_tokens[:N], per_token=True).mean(dim=1)

    return per_head_residuals, per_layer_residuals, loss
        
        

        

In [ ]:
#per_head, per_layer, loss = tokens_to_idim_prompt_and_free_decomposed_attn_heads(arxiv_tokens, 20)

Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results 

In [18]:
_, per_layer_arxiv, loss_arxiv = tokens_to_idim_prompt_and_free_decomposed_attn_heads(arxiv_tokens, 20)
_, per_layer_wiki, loss_wiki = tokens_to_idim_prompt_and_free_decomposed_attn_heads(wiki_tokens, 20)
_, per_layer_math, loss_math = tokens_to_idim_prompt_and_free_decomposed_attn_heads(math_tokens, 20)

Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results when they weren't cached. Computing head results now
Tried to stack head results 

In [19]:
per_head_residuals = t.cat(per_head, dim=1)

In [21]:
per_head_residuals.shape

torch.Size([145, 20, 512, 768])

In [25]:
ids_list = []

In [26]:

for idx in range(20):
  ids = compute_ids(per_head_residuals[1:, idx])
  # print(accumulated_residual.shape)
  # tensors.append(ids[:, 0, 1])
  ids = ids[:, 0, 1]
  ids = einops.rearrange(
      ids,
      "(layer head_index) -> layer head_index",
      layer=model.cfg.n_layers,
      head_index=model.cfg.n_heads,
  )
  print(ids.shape)
  ids_list.append(ids)




(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)
(12, 12)


In [27]:
ids_list

[array([[1.61, 4.25, 2.63, 2.13, 1.22, 4.29, 3.79, 4.15, 3.25, 3.6 , 4.16,
         4.61],
        [2.92, 2.33, 2.94, 3.18, 4.08, 4.14, 4.53, 3.65, 3.98, 2.91, 3.63,
         2.91],
        [4.79, 3.47, 3.81, 4.5 , 3.93, 2.95, 3.26, 3.39, 4.09, 3.15, 3.53,
         2.85],
        [2.93, 4.15, 4.27, 4.96, 3.94, 3.58, 4.07, 3.63, 4.04, 5.32, 3.61,
         3.43],
        [3.36, 4.03, 3.67, 5.39, 3.88, 2.74, 6.27, 5.2 , 3.57, 5.37, 3.62,
         5.39],
        [4.9 , 4.67, 3.1 , 4.52, 4.31, 4.58, 4.52, 5.83, 4.59, 5.28, 5.81,
         2.94],
        [5.91, 4.23, 3.46, 5.35, 3.24, 5.53, 3.26, 5.86, 3.92, 5.59, 3.2 ,
         4.76],
        [5.08, 5.27, 4.21, 4.36, 4.54, 4.61, 5.52, 5.32, 4.56, 4.61, 4.78,
         5.03],
        [5.93, 5.09, 5.15, 5.21, 3.53, 5.07, 4.47, 4.82, 5.07, 4.61, 5.6 ,
         4.84],
        [5.65, 5.42, 4.28, 5.29, 5.28, 4.31, 5.  , 5.77, 4.92, 4.32, 4.88,
         4.62],
        [4.56, 4.3 , 4.46, 4.15, 4.86, 5.1 , 4.43, 4.93, 5.15, 4.86, 4.97,
         3.74],

In [31]:
ids_list = np.stack(ids_list, axis=0)

In [35]:
loss = loss.detach().cpu().numpy()

In [25]:
import numpy as np
from scipy import stats
import warnings
def calculate_pearson_correlation_loss_ids(ids: np.ndarray, loss: np.ndarray) -> np.ndarray:
    """
    Calculates the Pearson correlation coefficient between a 1D loss array
    and slices of a 3D ids array.

    Args:
        ids: A 3D NumPy array of shape [N, L, H]. N represents the number
             of samples or observations.
        loss: A 1D NumPy array of shape [N], representing the loss values
              corresponding to the N samples.

    Returns:
        A 2D NumPy array of shape [L, H] where each element [l, h] contains
        the Pearson correlation coefficient between the loss array and the
        ids[:, l, h] slice. Returns NaN for correlations where one of the
        input vectors has zero standard deviation (is constant).

    Raises:
        ValueError: If the first dimension of 'ids' (N) does not match the
                    length of 'loss' (N).
        ValueError: If 'ids' is not a 3D array or 'loss' is not a 1D array.
    """
    # --- Input Validation ---
    if ids.ndim != 3:
        raise ValueError(f"Input 'ids' must be a 3D array, but got shape {ids.shape}")
    if loss.ndim != 1:
        raise ValueError(f"Input 'loss' must be a 1D array, but got shape {loss.shape}")

    N_ids, L, H = ids.shape
    N_loss = loss.shape[0]

    if N_ids != N_loss:
        raise ValueError(f"The first dimension of 'ids' ({N_ids}) must match "
                         f"the length of 'loss' ({N_loss})")

    if N_ids < 2:
         warnings.warn("Correlation is not defined for less than 2 samples. Returning NaNs.",
                       UserWarning)
         return np.full((L, H), np.nan)


    # --- Calculation ---
    # Initialize the result array
    correlations = np.zeros((L, H), dtype=float) # Use float for potential NaNs

    # Iterate through the L and H dimensions
    for l in range(L):
        for h in range(H):
            # Extract the slice corresponding to the current (l, h) position
            # This slice has shape (N,)
            ids_slice = ids[:, l, h]

            # Calculate Pearson correlation between the loss array and the current slice
            # stats.pearsonr returns (correlation_coefficient, p-value)
            # We only need the correlation coefficient (the first element)
            # It handles potential constant inputs (zero standard deviation) by returning NaN
            with warnings.catch_warnings():
                # Suppress RuntimeWarning for constant input vectors leading to NaN
                warnings.simplefilter("ignore", category=RuntimeWarning)
                corr, _ = stats.pearsonr(loss, ids_slice)

            # Store the correlation coefficient
            correlations[l, h] = corr

    return correlations

In [38]:
correlation = calculate_pearson_correlation_loss_ids(ids_list, loss)

In [40]:
correlation.shape

(12, 12)

In [41]:
from plot_utils import imshow

In [42]:
imshow(
    correlation,
    labels={"x": "Head", "y": "Layer"},
    title="Pearson Correlation, IDim and Loss",
)

In [44]:
per_layer[0].shape

torch.Size([26, 1, 512, 768])

In [19]:
per_layer_residuals_arxiv = t.cat(per_layer_arxiv, dim=1)
per_layer_residuals_wiki = t.cat(per_layer_wiki, dim=1)
per_layer_residuals_math = t.cat(per_layer_math, dim=1)

In [20]:
idim_per_layer_arxiv = []
idim_per_layer_wiki = []
idim_per_layer_math = []
for idx in range(20):
  ids = compute_ids(per_layer_residuals_arxiv[:, idx])
  idim_per_layer_arxiv.append(ids[1:, 0, 1])
  
  ids = compute_ids(per_layer_residuals_wiki[:, idx])
  idim_per_layer_wiki.append(ids[1:, 0, 1])
  
  ids = compute_ids(per_layer_residuals_math[:, idx])
  idim_per_layer_math.append(ids[1:, 0, 1])
  # line(ids[:, 0, 1],x=np.arange(model.cfg.n_layers * 2 + 1) / 2, hover_name=labels)

c:\Users\saepa\anaconda3\envs\arena-env\Lib\site-packages\dadapy\id_estimation.py:413: UserWarning: there may be data with zero distance from each other;
                this may compromise the correct behavior of some routines
  distances, dist_indices, mus, rs = self._return_mus_scaling(
c:\Users\saepa\anaconda3\envs\arena-env\Lib\site-packages\dadapy\id_estimation.py:413: UserWarning: there may be data with zero distance from each other;
                this may compromise the correct behavior of some routines
  distances, dist_indices, mus, rs = self._return_mus_scaling(
c:\Users\saepa\anaconda3\envs\arena-env\Lib\site-packages\dadapy\id_estimation.py:413: UserWarning: there may be data with zero distance from each other;
                this may compromise the correct behavior of some routines
  distances, dist_indices, mus, rs = self._return_mus_scaling(
c:\Users\saepa\anaconda3\envs\arena-env\Lib\site-packages\dadapy\id_estimation.py:413: UserWarning: there may be data with zero

In [21]:
idim_per_layer_arxiv = np.stack(idim_per_layer_arxiv, axis=0)
idim_per_layer_wiki = np.stack(idim_per_layer_wiki, axis=0)
idim_per_layer_math = np.stack(idim_per_layer_math, axis=0)

In [58]:
idim_per_layer.shape

(20, 25)

In [52]:
labels

['embed',
 'pos_embed',
 '0_attn_out',
 '0_mlp_out',
 '1_attn_out',
 '1_mlp_out',
 '2_attn_out',
 '2_mlp_out',
 '3_attn_out',
 '3_mlp_out',
 '4_attn_out',
 '4_mlp_out',
 '5_attn_out',
 '5_mlp_out',
 '6_attn_out',
 '6_mlp_out',
 '7_attn_out',
 '7_mlp_out',
 '8_attn_out',
 '8_mlp_out',
 '9_attn_out',
 '9_mlp_out',
 '10_attn_out',
 '10_mlp_out',
 '11_attn_out',
 '11_mlp_out']

In [46]:
per_layer_residuals.shape

torch.Size([26, 20, 512, 768])

In [53]:
loss.shape

(20,)

In [26]:
def calculate_pearson_correlation_loss_ids_1d(ids: np.ndarray, loss: np.ndarray) -> np.ndarray:
    """
    Calculates the Pearson correlation coefficient between a 1D loss array
    and each column of a 2D ids array.

    Args:
        ids: A 2D NumPy array of shape [N, L]. N represents the number
             of samples or observations, L represents the number of features/columns.
        loss: A 1D NumPy array of shape [N], representing the loss values
              corresponding to the N samples.

    Returns:
        A 1D NumPy array of shape [L] where each element [l] contains
        the Pearson correlation coefficient between the loss array and the
        ids[:, l] column (the l-th column of the ids array). Returns NaN
        for correlations where one of the input vectors has zero standard
        deviation (is constant).

    Raises:
        ValueError: If the first dimension of 'ids' (N) does not match the
                    length of 'loss' (N).
        ValueError: If 'ids' is not a 2D array or 'loss' is not a 1D array.
    """
    # --- Input Validation ---
    if ids.ndim != 2:
        raise ValueError(f"Input 'ids' must be a 2D array, but got shape {ids.shape}")
    if loss.ndim != 1:
        raise ValueError(f"Input 'loss' must be a 1D array, but got shape {loss.shape}")

    N_ids, L = ids.shape
    N_loss = loss.shape[0]

    if N_ids != N_loss:
        raise ValueError(f"The first dimension of 'ids' ({N_ids}) must match "
                         f"the length of 'loss' ({N_loss})")

    if N_ids < 2:
         warnings.warn("Correlation is not defined for less than 2 samples. Returning NaNs.",
                       UserWarning)
         return np.full(L, np.nan) # Return an array of NaNs with shape (L,)


    # --- Calculation ---
    # Initialize the result array
    correlations = np.zeros(L, dtype=float) # Use float for potential NaNs

    # Iterate through the L dimension (columns of ids)
    for l in range(L):
        # Extract the l-th column from ids
        # This slice has shape (N,)
        ids_slice = ids[:, l]

        # Calculate Pearson correlation between the loss array and the current column
        # stats.pearsonr returns (correlation_coefficient, p-value)
        # We only need the correlation coefficient (the first element)
        # It handles potential constant inputs (zero standard deviation) by returning NaN
        with warnings.catch_warnings():
            # Suppress RuntimeWarning for constant input vectors leading to NaN
            warnings.simplefilter("ignore", category=RuntimeWarning)
            corr, _ = stats.pearsonr(loss, ids_slice)

        # Store the correlation coefficient
        correlations[l] = corr

    return correlations


In [28]:
loss_arxiv = loss_arxiv.detach().cpu().numpy()
loss_math = loss_math.detach().cpu().numpy()
loss_wiki = loss_wiki.detach().cpu().numpy()

In [29]:
corr_arxiv = calculate_pearson_correlation_loss_ids_1d(idim_per_layer_arxiv, loss_arxiv)
corr_math = calculate_pearson_correlation_loss_ids_1d(idim_per_layer_math, loss_math)
corr_wiki = calculate_pearson_correlation_loss_ids_1d(idim_per_layer_wiki, loss_wiki)

In [61]:
corr

array([  nan,  0.61,  0.69,  0.5 ,  0.58,  0.6 ,  0.62,  0.69,  0.58,
        0.56,  0.58,  0.39,  0.5 , -0.17,  0.45,  0.24,  0.48, -0.17,
        0.43,  0.49,  0.44,  0.54,  0.46,  0.46,  0.41])

In [31]:
import plotly
import plotly.graph_objects as go
from plotly.colors import qualitative
colors = qualitative.Plotly # Get the default sequence
D3colors = qualitative.D3

In [34]:
# Initialize a Plotly Figure object
fig = go.Figure()



# Example: Get the first color


# Add the main trace: the average line with error bars
fig.add_trace(go.Scatter(
    x=labels,
    y=corr_arxiv,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Arxiv Prompt',   # Name for the legend
    line=dict(color=colors[0], width=2), # Style the average line
    marker=dict(size=5, color=colors[0]), # Style the markers

))
fig.add_trace(go.Scatter(
    x=labels,
    y=corr_wiki,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Wiki Prompt',   # Name for the legend
    line=dict(color=colors[1], width=2), # Style the average line
    marker=dict(size=5, color=colors[1]), # Style the markers

))
fig.add_trace(go.Scatter(
    x=labels,
    y=corr_math,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Math Prompt',   # Name for the legend
    line=dict(color=colors[2], width=2), # Style the average line
    marker=dict(size=5, color=colors[2]), # Style the markers

))
# --- Optional: Add the original individual lines for context (commented out by default) ---
# Uncomment the following loop if you want to see the underlying data lines.
# for i in range(num_lines):
#     fig.add_trace(go.Scatter(
#         x=x_values,
#         y=y_data[i, :],
#         mode='lines',
#         name=f'Line {i+1}',
#         line=dict(width=0.5, color='rgba(128, 128, 128, 0.4)'), # Thin, semi-transparent grey lines
#         showlegend=False # Hide these individual lines from the main legend
#     ))

# --- 4. Customize the Plot Layout ---
fig.update_layout(font=dict(size=22)) 
fig.update_layout(
    title='Pearson Correlation Layer and Loss', # Plot title
    xaxis_title='Layers',                 # X-axis label
    yaxis_title='Pearson Correlation',                               # Y-axis label
    legend_title='Prompt and IDim Type',                               # Title for the legend box
    hovermode='x unified',                               # Show hover info for all traces at a given x-value
    template='plotly_white',
    showlegend=True,
)

#fig.write_image('IDIM_prompt_categories.png')
# --- 5. Show the Plot ---

# Display the figure. This will typically open it in a browser window
# or display it in the output cell of a Jupyter notebook.
fig.show()
